# EDA na Prática — Dataset: FIFA World Cup, todas as partidas (1930–2026)

Réplica do ciclo de EDA apresentado em aula (dataset *Heart Disease*, UCI), aplicado ao dataset escolhido:
**"FIFA World Cup All Matches (1930–2026)"** (Kaggle, laxmikantaroy) — histórico de partidas de Copa do Mundo, com placares, sedes, fase da competição e método de resultado.

Assim como no exemplo do professor, seguimos o ciclo **iterativo**: estrutura → dados faltantes → estatística descritiva ("antes") → tratamento de ausentes/outliers → estatística descritiva ("depois") → princípio relacional (bivariada/multivariada) → síntese final.

Perguntas guiadas que não fazem sentido para este dataset (ex: as que dependem de um alvo clínico binário como "doença cardíaca") foram adaptadas ou substituídas por perguntas equivalentes fazendo sentido para dados de partidas de futebol (ex: "o time mandante/sede vence mais?", no lugar de "fairness entre sexo e faixa etária").


## Setup de Bibliotecas e Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

COR_NEUTRA = "lightgray"     # convenção: cinza-claro pra tudo que é "pano de fundo" genérico
COR_CONTEXTO = "#e9d4d0"    # rosa-claro pra pano de fundo / classe de referência
COR_DESTAQUE = "#2d223a"    # roxo pra classe de interesse / "o personagem da história"
COR_ALERTA = "tab:red"       # exceção deliberada: outliers/pontos de atenção


## Carregando o Dataset

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
arquivo_bruto = PROJECT_ROOT / "data/raw/fifa_world_cup_all_matches_1930_2026.csv"
if not arquivo_bruto.exists():
    raise FileNotFoundError("Coloque o CSV original do Kaggle em data/raw/. Consulte o README.")
df = pd.read_csv(arquivo_bruto)
display(df.head())


## 1. Princípio Estrutural

Antes de qualquer análise, vamos entender a "anatomia" dos dados: quantas linhas e colunas, tipos de dados, ausentes e duplicadas.

**Antes de rodar:** o dataset cobre 23 edições da Copa do Mundo (1930 a 2026, incluindo a atual em andamento). Cada Copa tem entre 18 (1930, com só 13 seleções) e 104 partidas (2026, com 48 seleções). Quantas linhas (partidas) vocês esperam no total — algumas centenas ou mais de mil?

### Dimensão e Tipos de Dados

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

**Resposta:** o dataset tem 1069 partidas e 23 colunas. Faz sentido: são 23 edições, com uma média de 46 partidas por Copa (variando de 18 em 1930 até 104 na expansão de 2026 para 48 seleções).


As colunas descrevem: identificação da partida (`match_id`, `world_cup_year`, `date`), contexto (`host_country`, `stage`, `round_raw`, `group`), os dois times (`team1`, `team2`), placares em diferentes momentos do jogo (`halftime_*`, `fulltime_*`, `extra_time_*`, `penalty_*`), o desfecho (`winner`, `result_method`) e o local (`stadium`, `city`). As duas últimas (`total_goals_team1`, `total_goals_team2`) parecem ser colunas derivadas — vamos investigar sua consistência mais adiante.

### Duplicadas

In [ ]:
df.duplicated().sum()

Nenhuma linha duplicada — esperado, já que `match_id` é único por partida.

### Criando uma variável de "resultado" para facilitar a interpretação

Diferente do Heart Disease (que tem um alvo clínico binário pronto), aqui não existe uma coluna de "classe" única. Vamos derivar uma variável categórica de interesse — o **resultado da partida sob a ótica do `team1`** — que vamos usar como eixo de comparação ao longo da aula (parecido com o papel que `doenca_cardiaca` teve no exemplo do professor).

In [ ]:
def resultado_time1(row):
    if row['winner'] == 'Draw':
        return 'Empate'
    elif row['winner'] == row['team1']:
        return 'Vitória Time Casa'
    else:
        return 'Vitória Time Fora'

df['resultado'] = df.apply(resultado_time1, axis=1)
df['resultado'].value_counts()


## 2. Dados Faltantes

In [ ]:
df.isna().sum()

**Discussão:** Alguma feature tem dados ausentes?

*Resposta:* sim, várias: `group` (290), `halftime_score_team1/2` (709 cada), `fulltime_score_team1/2` (1 cada), `extra_time_score_team1/2` (987 cada), `penalty_score_team1/2` (1030 cada) e `city` (1).

#### Identificando o mecanismo de dados ausentes

**Conhecimento de domínio**

- `group`: só existe fase de grupos em parte do formato do torneio. Toda partida de mata-mata (oitavas, quartas, semifinal, final etc.) não tem grupo. **Isso não é um dado que "deveria" existir e falhou em ser coletado — é estruturalmente inexistente** para esse tipo de linha. Mecanismo: **estrutural / MNAR determinístico** (a ausência depende 100% do valor de outra coluna, `stage`).
- `extra_time_score_*` e `penalty_score_*`: só existem quando a partida foi para prorrogação ou pênaltis. Mesma lógica do `group`: ausência **estrutural**, não um dado perdido.
- `halftime_score_*`: aqui é diferente — o placar do intervalo *deveria* existir para qualquer partida disputada, mas não foi registrado/disponibilizado para boa parte do histórico (ver padrão por ano abaixo). Isso é mais consistente com **MAR**: a ausência depende de uma variável observada (o ano/fonte de dados de cada Copa), não do próprio valor do placar.
- `fulltime_score_team1/2` (1 linha) e `city` (1 linha): candidatos a **caso isolado** — vamos investigar diretamente.


In [ ]:
df.groupby('world_cup_year')['halftime_score_team1'].apply(lambda s: s.isnull().mean()).round(2)

**Pra discutir:** o padrão de ausência do placar do intervalo não é uniforme ao longo do tempo — Copas de 1930 a 1938 têm o dado quase completo, depois some quase totalmente entre 1950 e 2002 (com exceções pontuais), e volta a aparecer de 2006 em diante. Isso é coerente com uma explicação de **domínio**: bases históricas de Copas antigas frequentemente não registram placar de intervalo, e diferentes provedores/atualizações da base preencheram esse campo de forma desigual por edição. É um exemplo de **MAR** (depende do ano/fonte, uma variável observada), não de **MNAR** (não depende do próprio placar do intervalo ter sido, digamos, incomum).

In [ ]:
df[df['fulltime_score_team1'].isnull()][['world_cup_year','team1','team2','stage','result_method']]

A única linha com placar de tempo integral ausente é **Suécia x Áustria, 1938**, com `result_method = Walkover` — a Áustria foi anexada pela Alemanha (Anschluss) pouco antes do torneio e se retirou, dando a vitória à Suécia sem a partida ser disputada. Ou seja, não é um erro de coleta: **o jogo não aconteceu**, então não existe placar para preencher. Mecanismo: **MNAR determinístico / estrutural** (a ausência é consequência direta e explicável do próprio evento).

In [ ]:
df[df['city'].isnull()]

O único registro sem cidade é a mesma partida (Walkover de 1938) — outro efeito colateral de a partida não ter sido realizada em um estádio real.

**Nota para quando comparado ao exemplo do Heart Disease:** lá, a ausência tinha que ser investigada com hipóteses (MAR vs MNAR) porque não havia certeza documental. Aqui, boa parte da ausência é **estrutural e determinística** (dá para provar olhando outra coluna, como `stage` ou `result_method`) — um lembrete de que nem todo dado faltante é incerteza: às vezes é o próprio desenho dos dados.

## 3. Princípio Distribucional

Ainda com os dados originais (sem tratar ausentes/outliers), vamos olhar a distribuição da variável quantitativa mais central do dataset: o **total de gols por partida** (`gols_totais = fulltime_score_team1 + fulltime_score_team2`). Esse retrato "antes" vai orientar as decisões de tratamento da próxima seção.

In [ ]:
df_antes = df.copy()
df_antes['gols_totais'] = df_antes['fulltime_score_team1'] + df_antes['fulltime_score_team2']
df_antes[['fulltime_score_team1','fulltime_score_team2','gols_totais']].describe()


In [ ]:
print("Skewness:", df_antes['gols_totais'].skew().round(2))
print("Kurtosis:", df_antes['gols_totais'].kurt().round(2))


**Pra discutir:** `gols_totais` tem assimetria (`skewness`) de aproximadamente **0.92** (à direita) e curtose de **1.36** — cauda mais pesada que uma normal, com poucas partidas de goleada puxando a distribuição. Isso é coerente com o que se esperaria de dados de contagem de gols em futebol: a maioria das partidas tem poucos gols (0 a 3), e só ocasionalmente aparece uma goleada.

### Distribuições de dados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(df_antes['gols_totais'].dropna(), bins=range(0,13), color=COR_DESTAQUE, ax=axes[0])
axes[0].set_title("Distribuição de Gols Totais por Partida")
axes[0].set_xlabel("Gols totais na partida")

sns.countplot(data=df_antes, x='resultado', order=['Vitória Time Casa','Empate','Vitória Time Fora'],
              color=COR_CONTEXTO, ax=axes[1])
axes[1].set_title("Distribuição do Resultado (Time 1 x Time 2)")

plt.tight_layout()
plt.show()


**Pra discutir:** o histograma de gols mostra a cauda direita esperada (maioria das partidas com 0 a 3 gols, poucas goleadas com 6+). Já o resultado sob a ótica do `team1` está relativamente equilibrado entre "Vitória Time 1" e "Vitória Time 2" — o que é esperado, já que `team1`/`team2` não representam mandante/visitante de forma consistente, apenas a ordem de registro da partida.

#### Testes de Aderência

*Será que o total de gols por partida segue uma distribuição normal?* Vamos usar Shapiro-Wilk (como no exemplo do professor) e apoiar com QQ-Plot.

In [ ]:
from scipy.stats import shapiro

gols_validos = df_antes['gols_totais'].dropna()
amostra = gols_validos.sample(500, random_state=42) if len(gols_validos) > 500 else gols_validos
stat, p = shapiro(amostra)
print(f"Shapiro-Wilk: estatística={stat:.4f}, p-valor={p:.2e}")


In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
stats.probplot(gols_validos, dist="norm", plot=ax)

ax.get_lines()[0].set_markerfacecolor(COR_CONTEXTO)
ax.get_lines()[0].set_markeredgecolor(COR_CONTEXTO)
ax.get_lines()[0].set_markersize(5)
ax.get_lines()[1].set_color(COR_ALERTA)
ax.get_lines()[1].set_linewidth(2)

ax.set_title("QQ-Plot: Gols Totais x Distribuição Normal")
plt.tight_layout()
plt.show()


**Como ler o QQ-Plot:** se `gols_totais` fosse normal, os pontos (quantis observados) cairiam em cima da linha vermelha (quantis teóricos de uma normal). O que vemos é o contrário: os pontos seguem a reta razoavelmente bem na região central (poucos gols), mas se afastam bastante dela nas caudas — principalmente na cauda superior, onde as goleadas empurram os quantis observados bem acima do que uma normal preveria. Isso é a mesma conclusão do Shapiro-Wilk, só que visual: o desvio de normalidade não é uniforme, é concentrado nas partidas de muitos gols (cauda pesada à direita), reforçando que uma Poisson (ou outra distribuição de contagem) descreve melhor esse fenômeno do que a Normal.

**Como ler:** o p-valor do Shapiro-Wilk é extremamente pequeno (< 0.001), rejeitando a hipótese de normalidade — temos **evidência estatística contra a normalidade**. O histograma confirma: a distribuição observada é assimétrica à direita e não acompanha bem a curva normal ajustada, especialmente na cauda superior (partidas com muitos gols). Isso é totalmente esperado: gols por partida são **dados de contagem** (mais próximos de uma distribuição Poisson do que Normal), então usar testes que assumem normalidade (Pearson, T-test, ANOVA) precisa de cautela — vamos priorizar os equivalentes não-paramétricos na análise bivariada.

Com n ≈ 1069 (amostra grande), o teste tem bastante poder — a rejeição não é um artefato de amostra pequena.

### Métricas de Tendência Central

In [ ]:
df_antes['gols_totais'].agg(['mean','median'])

In [ ]:
df_antes['gols_totais'].mode()

### Dispersão

In [ ]:
df_antes['gols_totais'].agg(['std','var'])


In [ ]:
q1 = df_antes['gols_totais'].quantile(0.25)
q3 = df_antes['gols_totais'].quantile(0.75)
iqr = q3 - q1
print(f"Q1={q1}, Q3={q3}, IQR={iqr}")


In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
sns.boxplot(data=df_antes, x='resultado', y='gols_totais',
            order=['Vitória Time Casa','Empate','Vitória Time Fora'],
            palette=[COR_CONTEXTO, COR_NEUTRA, COR_DESTAQUE], ax=ax)
ax.set_title("Gols Totais por Resultado (antes do tratamento)")
plt.show()


**Pra discutir:** a mediana de gols totais é **2**, com Q1=1 e Q3=3 (IQR=2). O boxplot por resultado mostra que empates concentram menos gols totais (esperado — muitos empates são 0x0 ou 1x1), enquanto vitórias (de qualquer lado) têm caixas ligeiramente mais altas — indício, antes mesmo da bivariada formal, de que "quantidade de gols" e "resultado decisivo x empate" têm alguma relação.

## 4. Tratamento de Dados Ausentes e Outliers

Com o retrato "antes" em mãos, já temos informação suficiente pra decidir como tratar tanto os ausentes quanto os outliers.

### Detecção de Outliers por IQR

In [ ]:
lim_inf = q1 - 1.5*iqr
lim_sup = q3 + 1.5*iqr
outliers = df_antes[(df_antes['gols_totais'] < lim_inf) | (df_antes['gols_totais'] > lim_sup)]
print(f"Limite: [{lim_inf}, {lim_sup}]  |  Nº de outliers: {len(outliers)} ({len(outliers)/len(df_antes):.1%} das partidas)")
outliers[['world_cup_year','team1','team2','fulltime_score_team1','fulltime_score_team2','gols_totais']].sort_values('gols_totais', ascending=False).head(10)


### Visualizando os Outliers

In [ ]:
fig, ax = plt.subplots(figsize=(9,3.5))
sns.boxplot(x=df_antes['gols_totais'], color=COR_CONTEXTO, ax=ax)
sns.stripplot(x=outliers['gols_totais'], color=COR_ALERTA, size=6, jitter=0.15, ax=ax)
ax.set_title("Outliers de Gols Totais por Partida (IQR)")
plt.show()


**Pra discutir:** olhando os jogos com mais gols (ex: Hungria 10x1 El Salvador em 1982, Alemanha 8x0 Arábia Saudita, Brasil 6x5 Polônia em 1938), eles são resultados **reais e documentados**, não erros de digitação — apenas goleadas raras entre seleções de níveis técnicos muito diferentes. Classificação (usando a mesma tabela de 3 categorias do exemplo): **outlier estatístico legítimo**. Tratamento recomendado: **manter** — remover essas partidas distorceria a história real do torneio, e são exatamente esses jogos que interessam em qualquer análise histórica de futebol.

### Decidindo o Tratamento dos Dados Ausentes

Reunindo o que discutimos na Seção 2:

| Coluna | Mecanismo | Decisão | Justificativa |
|---|---|---|---|
| `group` | Estrutural (MNAR determinístico) | Recodificar `NaN` → `"Fase Eliminatória"` | Ausência não é falta de coleta, é ausência de conceito (mata-mata não tem grupo) |
| `extra_time_score_*` | Estrutural | Preencher com `0` | Ausência representa "não houve prorrogação", que equivale a 0 gols na prorrogação |
| `penalty_score_*` | Estrutural | Preencher com `0` | Mesma lógica — "não foi a pênaltis" |
| `halftime_score_*` | MAR (depende do ano/fonte) | **Manter `NaN`**, não imputar | Não temos como recuperar um placar histórico não registrado; imputar inventaria dado. Análises que precisem do intervalo devem usar apenas o subconjunto completo |
| `fulltime_score_*` (1 linha) | MNAR determinístico (walkover) | Excluir a linha das análises quantitativas de gols; manter no dataset para contagem de partidas | O jogo não ocorreu — não existe gol a ser preenchido |
| `city` (1 linha) | Mesmo caso do walkover | Preencher com `"N/A (Walkover)"` | Consistência de rótulo, sem inventar um estádio |


In [ ]:
df['group'] = df['group'].fillna('Fase Eliminatória')
df['extra_time_score_team1'] = df['extra_time_score_team1'].fillna(0)
df['extra_time_score_team2'] = df['extra_time_score_team2'].fillna(0)
df['penalty_score_team1'] = df['penalty_score_team1'].fillna(0)
df['penalty_score_team2'] = df['penalty_score_team2'].fillna(0)
df['city'] = df['city'].fillna('N/A (Walkover)')

# gols_totais recalculado; a partida sem placar (walkover) fica NaN e é excluída SÓ das
# estatísticas de gols, mas continua no df principal
df['gols_totais'] = df['fulltime_score_team1'] + df['fulltime_score_team2']

df.isnull().sum()


### Decidindo o Tratamento dos Outliers

Como concluímos acima, os outliers de gols são **estatisticamente legítimos** (goleadas reais). Diferente do caso de dados ausentes, aqui **não vamos remover nem imputar** — vamos manter os valores originais. Só para fins didáticos (igual ao exemplo do Heart Disease com `chol_log`), vamos ilustrar como ficaria uma transformação logarítmica em `gols_totais`, caso fosse necessário alimentar algum teste paramétrico mais sensível à assimetria no futuro.

In [ ]:
df['gols_totais_log'] = np.log1p(df['gols_totais'])

fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
sns.histplot(df['gols_totais'].dropna(), color=COR_CONTEXTO, ax=axes[0])
axes[0].set_title("Gols Totais (original)")
sns.histplot(df['gols_totais_log'].dropna(), color=COR_DESTAQUE, ax=axes[1])
axes[1].set_title("Gols Totais (log1p)")
plt.tight_layout()
plt.show()


**Pra discutir:** a transformação log reduz visivelmente a assimetria (a cauda direita fica bem mais curta). Mantemos `gols_totais_log` só como demonstração — as próximas seções seguem com os valores originais, já que não há, por ora, nenhum teste paramétrico que exija essa transformação.

**Nota:** as colunas `total_goals_team1`/`total_goals_team2` originais do dataset (diferentes de `gols_totais`, que calculamos a partir de `fulltime_score`) merecem um alerta: ao comparar as duas, elas batem em menos de metade das partidas. Isso é um indício de **erro de dados** (terceira categoria da tabela de outliers) nessas duas colunas específicas — por isso optamos por não usá-las e recalcular `gols_totais` a partir dos placares de tempo integral, que são mais consistentes e verificáveis.

In [ ]:
bate = (df['total_goals_team1'] == df['fulltime_score_team1']).mean()
print(f"'total_goals_team1' bate com o placar de tempo integral em apenas {bate:.1%} das partidas — por isso não usamos essa coluna na análise.")


## 5. Estatística Descritiva pt 2

Fechando o ciclo iterativo: tratamos os ausentes (recodificando estruturais e preservando o único caso genuinamente ausente) e decidimos manter os outliers. Rodamos a mesma bateria de estatísticas da Seção 3 para comparar com o "antes".

In [ ]:
comparacao = pd.DataFrame({
    'antes': df_antes['gols_totais'].describe(),
    'depois': df['gols_totais'].describe(),
})
comparacao


**Pra discutir:** os valores praticamente não mudam (a diferença de `count` é de apenas 1 linha, a partida walkover). Isso é esperado — o tratamento foi conservador por decisão: **não alteramos nenhum outlier** e só removemos 1 de 1069 linhas por não ter placar. Pouca mudança aqui **confirma** que o tratamento foi proporcional ao problema, não que ele "não fez nada".

In [ ]:
stat, p = shapiro(df['gols_totais'].dropna().sample(500, random_state=42))
print(f"Shapiro-Wilk (depois): estatística={stat:.4f}, p-valor={p:.2e}")


O total de gols continua rejeitando normalidade — como esperado, já que não tratamos nenhum outlier estatístico (eles são legítimos).

## 6. Princípio Relacional

### Análise de Variáveis Qualitativas

#### Frequências absolutas e relativas

In [ ]:
df['stage'].value_counts()

In [ ]:
df['stage'].value_counts(normalize=True).round(3)

#### Visualizando as frequências

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ordem = df['stage'].value_counts().index
sns.countplot(data=df, y='stage', order=ordem, color=COR_DESTAQUE, ax=ax)
ax.set_title("Frequência de Partidas por Fase")
plt.tight_layout()
plt.show()


**Pra discutir:** `Group Stage` domina (768 de 1069 partidas, ~72%) — bem esperado, já que a fase de grupos sempre tem mais jogos que qualquer fase eliminatória isolada. Isso é um desbalanceamento **estrutural do formato do torneio**, não um problema de coleta, mas precisa ser levado em conta ao comparar "gols por fase": a fase de grupos, sozinha, vai dominar qualquer média geral.

#### Desbalanceamento entre Classes

In [ ]:
def razao_desbalanceamento(serie):
    contagem = serie.value_counts()
    return contagem.max() / contagem.min()

def entropia_normalizada(serie):
    p = serie.value_counts(normalize=True)
    n = len(p)
    if n <= 1:
        return 1.0
    return (-(p * np.log(p)).sum()) / np.log(n)

for col in ['stage', 'result_method', 'resultado', 'host_country']:
    print(f"{col:15s} | razão desbalanceamento: {razao_desbalanceamento(df[col]):7.1f} | entropia normalizada: {entropia_normalizada(df[col]):.3f}")


**Pra discutir:** `host_country` é, de longe, a mais desbalanceada (entropia baixa) — poucos países sediaram a Copa mais de uma vez (Brasil, México, Alemanha, França, Itália e Estados Unidos aparecem repetidas vezes, enquanto a maioria sediou uma única edição). `resultado` (Vitória Time 1 / Empate / Vitória Time 2) é a mais equilibrada, como já era visível no histograma da Seção 3 — o que faz sentido, já que "time1" e "time2" são só a ordem de registro da partida, sem significado de mandante/visitante.

### Análise Bivariada

#### Quantitativa x Quantitativa — Correlação

Já vimos que `gols_totais` não passa em Shapiro-Wilk, então Spearman é a escolha mais segura — mas calculamos as três (Pearson, Spearman, Kendall) para comparar, igual ao exemplo do professor.

In [ ]:
num_cols = ['world_cup_year', 'fulltime_score_team1', 'fulltime_score_team2', 'gols_totais']
corr_pearson = df[num_cols].corr(method='pearson')
corr_spearman = df[num_cols].corr(method='spearman')
corr_kendall = df[num_cols].corr(method='kendall')

print("Pearson (world_cup_year x gols_totais):", corr_pearson.loc['world_cup_year','gols_totais'].round(3))
print("Spearman (world_cup_year x gols_totais):", corr_spearman.loc['world_cup_year','gols_totais'].round(3))
print("Kendall (world_cup_year x gols_totais):", corr_kendall.loc['world_cup_year','gols_totais'].round(3))


#### Dispersão (scatter) entre duas variáveis quantitativas

Escolhemos `world_cup_year` x `gols_totais` por um motivo de domínio: existe uma narrativa conhecida no futebol de que o jogo ficou mais defensivo/tático ao longo das décadas, então vale checar se a média de gols por partida realmente caiu historicamente.

In [ ]:
media_por_ano = df.groupby('world_cup_year')['gols_totais'].mean()

fig, ax = plt.subplots(figsize=(10,5))
sns.scatterplot(x=df['world_cup_year'], y=df['gols_totais'], color=COR_CONTEXTO, alpha=0.4, ax=ax)
sns.lineplot(x=media_por_ano.index, y=media_por_ano.values, color=COR_DESTAQUE, marker='o', linewidth=2, ax=ax)
ax.set_title("Gols Totais por Partida ao Longo das Copas (linha = média por edição)")
ax.set_xlabel("Ano da Copa")
ax.set_ylabel("Gols totais na partida")
plt.show()


**Pra discutir:** a correlação entre ano e gols totais é fraca e negativa (|r| bem abaixo de 0.3 nas três métricas) — ou seja, **não existe uma tendência linear forte** de queda de gols ao longo do tempo, apesar da percepção popular de "futebol mais defensivo hoje". A média por edição (linha roxa) oscila bastante entre Copas, sem uma tendência monotônica clara — mais um exemplo de como a percepção popular nem sempre é confirmada pelos dados.

#### Quantitativa x Qualitativa: Comparando Grupos

Como `gols_totais` não é normal (Seção 3), usamos o equivalente não-paramétrico. Com `stage` tendo várias categorias, comparamos ANOVA (paramétrica) com **Kruskal-Wallis** (não-paramétrico), mantendo a ANOVA só como contraste pedagógico.

In [ ]:
grupos_stage = [g['gols_totais'].dropna().values for _, g in df.groupby('stage') if len(g) >= 5]

anova_stat, anova_p = stats.f_oneway(*grupos_stage)
kw_stat, kw_p = stats.kruskal(*grupos_stage)

print(f"ANOVA:           estatística={anova_stat:.3f}, p-valor={anova_p:.4f}")
print(f"Kruskal-Wallis:  estatística={kw_stat:.3f}, p-valor={kw_p:.4f}")


**Pra discutir:** os dois testes concordam que existe diferença significativa de gols entre as fases (p < 0.05 nos dois). Como já sabemos que `gols_totais` não é normal, o **Kruskal-Wallis** é o resultado que deveríamos reportar como principal — a ANOVA fica só como contraste. Isso é coerente com o que vimos no boxplot: fases finais (`Final`, `Third Place Play-off`) tendem a ter medianas de gols um pouco mais altas que a fase de grupos e as rodadas eliminatórias mais cedo (`Round of 16`, `Quarter-final`), embora a diferença não seja enorme.

#### Qualitativa x Qualitativa: mandante/sede vence mais? (equivalente ao exercício de "fairness" do exemplo)

No dataset Heart Disease, a discussão de fairness perguntava se o desfecho (doença) se distribui igual entre grupos protegidos (sexo, idade). Aqui, a pergunta equivalente e natural é: **o país-sede tem vantagem de jogar em casa?** Vamos comparar a taxa de vitória do país-sede contra a taxa de vitória "esperada" caso não houvesse vantagem alguma (um sorteio justo entre dois lados, ignorando empates).

In [ ]:
df['time_e_sede'] = np.where(df['team1'] == df['host_country'], 'team1',
                       np.where(df['team2'] == df['host_country'], 'team2', 'nenhum'))

jogos_com_sede = df[df['time_e_sede'] != 'nenhum'].copy()
jogos_com_sede['sede_venceu'] = np.where(
    ((jogos_com_sede['time_e_sede']=='team1') & (jogos_com_sede['winner']==jogos_com_sede['team1'])) |
    ((jogos_com_sede['time_e_sede']=='team2') & (jogos_com_sede['winner']==jogos_com_sede['team2'])),
    'Sede Venceu',
    np.where(jogos_com_sede['winner']=='Draw', 'Empate', 'Sede Perdeu')
)

print(f"Partidas em que o país-sede jogou: {len(jogos_com_sede)}")
jogos_com_sede['sede_venceu'].value_counts(normalize=True).round(3)


In [ ]:
tabela_contingencia = pd.crosstab(jogos_com_sede['sede_venceu'], columns='contagem')
chi2, p_chi2, dof, esperado = stats.chi2_contingency(
    pd.crosstab(jogos_com_sede['sede_venceu'], [1]*len(jogos_com_sede))
)
# Comparação direta contra uma distribuição uniforme hipotética entre os 3 desfechos
observado = jogos_com_sede['sede_venceu'].value_counts()
esperado_uniforme = [len(jogos_com_sede)/3]*3
chi2_uniforme, p_uniforme = stats.chisquare(observado, f_exp=esperado_uniforme)
print(f"Qui-quadrado (contra distribuição uniforme entre os 3 desfechos): estatística={chi2_uniforme:.2f}, p-valor={p_uniforme:.2e}")


**Pra discutir:** o país-sede venceu cerca de **68%** das partidas em que jogou — bem acima de um cenário "sem vantagem nenhuma". O Qui-Quadrado confirma que essa distribuição está longe de ser uniforme entre os três desfechos possíveis (vitória, empate, derrota). Assim como no exemplo de fairness do Heart Disease, o teste estatístico aponta **que** existe uma associação forte entre "jogar em casa" e "vencer", mas não explica sozinho **por quê** — pode ser vantagem real de torcida/adaptação/clima, mas também pode refletir que sedes historicamente tendem a ser selecionados entre países fortes no futebol (viés de seleção de quem sedia a Copa). A causa exige conhecimento de domínio, não só o número.

#### Outliers bivariados

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
sns.scatterplot(data=df, x='fulltime_score_team1', y='fulltime_score_team2', color=COR_CONTEXTO, alpha=0.5, ax=ax)
extremos = df[(df['fulltime_score_team1'] >= 6) | (df['fulltime_score_team2'] >= 6)]
sns.scatterplot(data=extremos, x='fulltime_score_team1', y='fulltime_score_team2', color=COR_ALERTA, s=80, ax=ax)
ax.set_title("Placar Time 1 x Placar Time 2 (destaque: goleadas)")
plt.show()


**Pra discutir:** diferente dos histogramas univariados (que olham cada placar separadamente), esse scatter mostra que praticamente todas as goleadas acontecem com **um dos lados fazendo 0 ou 1 gol** — ou seja, não existe (no histórico) uma combinação estranha do tipo "6x5" acontecendo com frequência; a maioria dos jogos de muitos gols é bem unilateral. O único ponto fora desse padrão (Brasil 6x5 Polônia, 1938) já havia aparecido como outlier univariado na Seção 4 — ele é reforçado, e não descoberto, pela visão bivariada.

### Análise Multivariada

#### Matriz de Correlação

In [ ]:
cols_numericas = ['world_cup_year', 'fulltime_score_team1', 'fulltime_score_team2',
                   'extra_time_score_team1', 'extra_time_score_team2',
                   'penalty_score_team1', 'penalty_score_team2', 'gols_totais']
matriz_corr = df[cols_numericas].corr(method='spearman')
matriz_corr.round(2)


#### Correlograma

In [ ]:
fig, ax = plt.subplots(figsize=(8,6.5))
mask = np.triu(np.ones_like(matriz_corr, dtype=bool))
sns.heatmap(matriz_corr, mask=mask, annot=True, fmt=".2f", cmap="RdPu",
            vmin=-1, vmax=1, ax=ax, cbar_kws={'label':'Spearman'})
ax.set_title("Correlograma (Spearman) — Variáveis Numéricas")
plt.tight_layout()
plt.show()


**Como interpretar:** |r| < 0.3 ~ fraca, 0.3 < |r| < 0.7 ~ moderada, |r| > 0.7 ~ forte.

**Pra discutir:** como esperado, `fulltime_score_team1`/`team2` têm correlação forte com `gols_totais` (afinal, `gols_totais` é a soma delas — redundância intencional, não um problema). Fora essa relação óbvia, nenhuma outra correlação passa de moderada: `world_cup_year` não se correlaciona de forma relevante com nenhum placar, reforçando a conclusão da análise bivariada de que não há uma tendência histórica forte de aumento ou queda de gols. `extra_time_score` e `penalty_score` também não se correlacionam fortemente com o placar de tempo integral, o que faz sentido — prorrogação e pênaltis só entram em jogo justamente quando o tempo normal termina empatado (score baixo/equilibrado), então a relação entre eles é mais sobre "quando acontece" do que "quanto vale".

## Síntese Final

### 1. Principais conclusões

1. **Estrutura e ausência são, em grande parte, previsíveis pelo formato do torneio.** A maioria dos dados "faltantes" (`group`, `extra_time_score`, `penalty_score`) não é incerteza estatística — é ausência estrutural, decorrente de regras do próprio torneio (nem toda partida tem grupo, nem toda partida vai a pênaltis). Só `halftime_score` mostrou um padrão real de MAR ligado à cobertura de dados por edição.
2. **Gols por partida seguem uma distribuição assimétrica e não-normal (tipo contagem/Poisson)**, com mediana de 2 gols totais e uma cauda direita puxada por goleadas raras, porém reais e documentadas — todas mantidas como outliers estatísticos legítimos.
3. **A narrativa popular de "futebol cada vez mais defensivo" não é fortemente confirmada pelos dados**: a correlação entre ano da Copa e gols por partida é fraca, sem tendência linear clara ao longo de quase 100 anos de torneio.
4. **Jogar em casa parece ser uma vantagem real e estatisticamente significativa**: o país-sede venceu ~68% das partidas em que participou, uma distribuição bem distante do que se esperaria sem vantagem alguma — embora, como em qualquer discussão de fairness, o teste não prove sozinho a causa (vantagem real de jogar em casa x viés de quais países tendem a sediar o torneio).
5. **Duas colunas do dataset original (`total_goals_team1`/`team2`) se mostraram inconsistentes** com o placar de tempo integral em mais da metade das partidas — um lembrete prático de que "ter uma coluna pronta" não dispensa validação cruzada com outras colunas antes de usá-la.

### 2. Declaração de Uso de IA

- **Ferramenta utilizada:** Claude (Anthropic), via chat com acesso a ambiente de execução de código (Python/pandas/scipy/matplotlib/seaborn).
- **Para o que foi aplicada:** (a) apoio na importação do dataset do Kaggle no Google Colab (kagglehub); (b) exploração inicial dos dados (estrutura, ausentes, inconsistências) para embasar as decisões de tratamento; (c) geração do código de cada etapa da EDA (estatísticas, testes de hipótese, visualizações) replicando a estrutura apresentada em aula; (d) redação de parte do texto de discussão de cada seção.
- **O que foi aceito, modificado ou rejeitado, e por quê:** as decisões de tratamento (ex: recodificar `group` ausente como "Fase Eliminatória" em vez de remover linhas, preencher `extra_time`/`penalty` com 0 em vez de deixar como ausente, manter outliers de gols como legítimos) foram revisadas e validadas manualmente, comparando com o raciocínio do exemplo do Heart Disease visto em aula, antes de aceitar o código gerado. A descoberta da inconsistência nas colunas `total_goals_team1/2` foi verificada numericamente (comparação direta com `fulltime_score`) antes de aceitar a conclusão de que a coluna era pouco confiável, em vez de aceitar essa afirmação apenas por ter sido sugerida.


## Exportação para a aplicação
A leitura foi adaptada para o CSV local. As células de análise e os tratamentos da EDA original foram preservados. O arquivo bruto permanece em data/raw/ e os dados tratados são exportados para data/clean/dataset.csv.

In [ ]:
pasta_tratada = PROJECT_ROOT / "data/clean"
pasta_tratada.mkdir(parents=True, exist_ok=True)
df.to_csv(pasta_tratada / "dataset.csv", index=False)
print(f"Dataset exportado: {len(df)} registros, {len(df.columns)} colunas.")
